# Pytorch batched training


Explore some more features on the (very small) Diabetes dataset: Mainly batched training, 

In [1]:
# complete imports if needed
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

The following block of code only is valid if you want to run the calculations on GPU - for this to be available you have to use some optional packages in the uv environment, which is very version and architecture dependent (e.g. `uv sync --extra cu126`). Let me know if you want to try any of this.

Additionally, this will impact some commands later on: The model and the data need to be passed specifically to the device (`.to(device)`) - for general compliance, I have commented those out.

In [2]:
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)

In [2]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

(442, 1)
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Use the dataloader to split the dataset into smaller batches (have a look how different batch sizes compare). Hint: Don't go for too small batch sizes, it will massively slow down the training!

In [4]:
from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            dtype=torch.float32 # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            dtype=torch.float32 # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)

In [5]:
train_dataset = MyDataFromDF(X_train, y_train)
test_dataset  = MyDataFromDF(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

Define the NN according to your first attempts.

In [49]:
class DiabetesNN(nn.Module):
    # define the layers
    def __init__(self, input_size, hidden_size, output_size, drop_out):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.dropout1 = nn.Dropout(p=drop_out) # only parts of the neurons are used.

        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.bn2 = nn.BatchNorm1d(hidden_size)
        self.dropout2 = nn.Dropout(p=drop_out)

        self.fc3 = nn.Linear(hidden_size, int(hidden_size/2))  # one hidden layer, try to add another one
        self.dropout3 = nn.Dropout(p=drop_out)

        self.fc4 = nn.Linear(int(hidden_size/2), output_size)  # output layer
    
    # Define how the data flows through the layers
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = self.dropout1(x)

        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        x = self.dropout2(x)

        x = torch.relu(self.fc3(x))  # ReLU activation function for hidden layer
        x = self.dropout3(x)
        
        x = self.fc4(x)   # no activation function for the output layer (because it is a regresion model!)
        return x

In [50]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 256
output_size = 1

drop_out=0.3 # specifies how much of the NN remains randomly inactive
learning_rate = 0.009
weight_decay = 0.0001

# Number of training iterations
num_epochs = 300

In [51]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size, drop_out)
# model = DiabetesNN(input_size, hidden_size, output_size, drop_out).to(device)

criterion = nn.MSELoss()  # loss function defined;

optimizer = optim.AdamW(model.parameters(), learning_rate, weight_decay=weight_decay) # gradient descent method based on average and squares of gradient

Train your model. Note how the defined batches need some adjustments in the code (evaluating loss...)

In [52]:
# batched training
for epoch in range(num_epochs):

    model.train()
    total_loss = 0
    total_samples = 0

    for data, targets in train_loader:
        data = data
        targets = targets
        # data = data.to(device)
        # targets = targets.to(device)

        outputs = model(data.to(torch.float32))

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    avg_loss = total_loss / total_samples

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [10/300], Loss: 3473.6568
Epoch [20/300], Loss: 3506.2056
Epoch [30/300], Loss: 3309.3557
Epoch [40/300], Loss: 3250.9381
Epoch [50/300], Loss: 3305.8609
Epoch [60/300], Loss: 3119.0129
Epoch [70/300], Loss: 3242.2366
Epoch [80/300], Loss: 3154.7418
Epoch [90/300], Loss: 2935.1359
Epoch [100/300], Loss: 3061.8495
Epoch [110/300], Loss: 2936.7337
Epoch [120/300], Loss: 2735.4778
Epoch [130/300], Loss: 3107.1581
Epoch [140/300], Loss: 2577.5280
Epoch [150/300], Loss: 2886.3691
Epoch [160/300], Loss: 2894.2582
Epoch [170/300], Loss: 2806.3666
Epoch [180/300], Loss: 2785.8053
Epoch [190/300], Loss: 2822.1003
Epoch [200/300], Loss: 2754.4214
Epoch [210/300], Loss: 3055.3490
Epoch [220/300], Loss: 2847.4314
Epoch [230/300], Loss: 2798.0173
Epoch [240/300], Loss: 2957.3343
Epoch [250/300], Loss: 2789.0876
Epoch [260/300], Loss: 2738.8406
Epoch [270/300], Loss: 2920.1728
Epoch [280/300], Loss: 3000.9109
Epoch [290/300], Loss: 2698.4795
Epoch [300/300], Loss: 2673.2991


Evaluate the model using the test set and see if the NN shows over or underfitting. Adjust the model accordingly or train again.

In [53]:
# Evaluate the model
model.eval()  # set model to eval mode (e.g. no dropout)

all_preds = []
all_targets = []

total_samples = 0
total_loss = 0

criterion = nn.MSELoss()

with torch.no_grad():
    for data, targets in test_loader:
        data = data
        targets = targets
        # data = data.to(device)
        # targets = targets.to(device)

        outputs = model(data.to(torch.float32))
        loss = criterion(outputs, targets)

        # accumulate loss over batches
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size # multiply by batch size to 
        total_samples += targets.size(0)
        
        # Predictions      
        probs = torch.sigmoid(outputs)     # convert for metrics
        preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().tolist()) # cpu() moves the tensor to cpu (if on gpu), tolist converts it to python list
        all_targets.extend(targets.cpu().tolist())

avg_loss = total_loss / total_samples

print(f"Average test Loss: {avg_loss:.4f}")


Average test Loss: 2619.5757


## Early stopping: 
Use a loop to determine epochs (Modified from snippet generated using ChatGPT).

In [55]:
def train_model(model, train_loader, test_loader, criterion, optimizer,
                max_epochs=100, patience=10):

    best_val_loss = float("inf")
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(max_epochs):
        # Training
        model.train()
        train_loss = 0.0

        for x, y in train_loader:

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for x, y in test_loader:
                
                outputs = model(x)
                loss = criterion(outputs, y)
                val_loss += loss.item()

        val_loss /= len(test_loader)

        print(f"Epoch {epoch+1}: train={train_loss:.4f}, val={val_loss:.4f}")

        # ---- Early stopping logic ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict()
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model

In [56]:
train_model(model, train_loader, test_loader, criterion, optimizer,
                max_epochs=100, patience=10)

Epoch 1: train=2665.7065, val=2452.5386
Epoch 2: train=2514.6313, val=2440.3989
Epoch 3: train=2687.5280, val=2465.4244
Epoch 4: train=2659.5919, val=2559.7338
Epoch 5: train=2663.2807, val=2486.3441
Epoch 6: train=2669.5665, val=2539.9709
Epoch 7: train=2725.3400, val=2517.8263
Epoch 8: train=2588.9041, val=2520.6951
Epoch 9: train=3032.1984, val=2533.4807
Epoch 10: train=2684.6086, val=2563.3027
Epoch 11: train=2692.8871, val=2516.5897
Epoch 12: train=2922.1816, val=2790.7865
Early stopping triggered at epoch 12


DiabetesNN(
  (fc1): Linear(in_features=10, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (dropout3): Dropout(p=0.3, inplace=False)
  (fc4): Linear(in_features=128, out_features=1, bias=True)
)